In [ ]:
from typing import Literal

from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel
from langchain_mistralai import ChatMistralAI
from pydantic import BaseModel, Field

load_dotenv()

True

In [3]:
model= ChatMistralAI(model="mistral-medium-2508")

# Simple chain

In [7]:
prompt= PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

parser= StrOutputParser()

chain= prompt | model | parser

result= chain.invoke({'topic': 'Jupiter'})
print(result)

Here are five fascinating facts about **Jupiter**, the largest planet in our solar system:

1. **Jupiter Could Have Been a Star (Almost!)**
   Jupiter is made mostly of hydrogen and helium—just like the Sun. If it had been about **80 times more massive**, nuclear fusion could have ignited in its core, turning it into a **red dwarf star** instead of a planet.

2. **The Great Red Spot is a Giant Storm**
   Jupiter’s iconic **Great Red Spot** is a massive storm that has been raging for **at least 400 years** (since it was first observed in 1665). It’s so large that **three Earths** could fit inside it, though it has been shrinking over time.

3. **Jupiter Has the Shortest Day in the Solar System**
   Despite its enormous size, Jupiter rotates **extremely fast**—completing a full rotation in just **9 hours and 55 minutes**. This rapid spin causes its equator to bulge and its poles to flatten, giving it an oblate shape.

4. **Jupiter Acts as a Cosmic Vacuum Cleaner**
   Its **strong gravita

In [16]:
prompt1= PromptTemplate(
    template='Generate adetailed report on {topic}',
    input_variables=['topic']
)

prompt2= PromptTemplate(
    template='Generate a 2 sentence summary from the following text\n{text}',
    input_variables=['text']
)
parser= StrOutputParser()

chain= prompt1 | model | parser | prompt2 | model | parser

result= chain.invoke({'topic': 'Jupiter'})
print(result)

Jupiter, the largest planet in our Solar System, is a **gas giant** with a turbulent atmosphere, a massive magnetic field, and **95 moons**, including **Europa and Ganymede**, which may harbor subsurface oceans capable of supporting life. Its **gravitational influence** shapes the Solar System by deflecting comets and asteroids, while ongoing missions like **Juno, Europa Clipper, and JUICE** continue to uncover its mysteries.


# Parellel chain

In [17]:
model1= ChatMistralAI(model="mistral-medium-latest")
model2= ChatMistralAI(model="mistral-small-latest")


In [21]:
text= """ 
Artificial intelligence (AI) is rapidly transforming the healthcare industry by enhancing diagnostic accuracy, personalizing treatment plans, and streamlining administrative tasks. Machine learning algorithms can analyze vast amounts of medical imaging data, such as MRIs and CT scans, to detect diseases like cancer at earlier, more treatable stages than ever before. Furthermore, AI-powered predictive analytics enable doctors to foresee patient risks, optimizing preventative care and reducing hospital readmission rates. Despite these advancements, the integration of AI faces challenges, including data privacy concerns, the need for high-quality, unbiased datasets, and the necessity of maintaining a human-centric approach to patient care. Therefore, while AI holds immense potential to improve patient outcomes, its adoption requires careful ethical oversight and validation to ensure safety and equity in medical treatment."""

In [24]:
prompt1= PromptTemplate(
    template='Generate simple notes from the following text:\n{text}',
    input_variables=['text']
)

prompt2= PromptTemplate(
    template='Generate 5 Question Answers from the following text:\n{text}',
    input_variables=['text']
)

prompt3= PromptTemplate(
    template='Merge the provided notes and quiz intoo a single document\n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes','quiz']
)

parser= StrOutputParser()

parallel_chain= RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain= prompt3 | model1 | parser

chain= parallel_chain | merge_chain

result= chain.invoke({'text': text})

print(result)

# **AI in Healthcare: Notes & Quiz**

---

## **Notes on AI in Healthcare**

### **Benefits of AI in Healthcare:**
- **Improved Diagnostics:**
  - Machine learning analyzes medical imaging (MRIs, CT scans) to detect diseases (e.g., cancer) earlier.
- **Personalized Treatment:**
  - AI tailors treatment plans based on patient data.
- **Predictive Analytics:**
  - Helps doctors predict patient risks, improving preventive care and reducing hospital readmissions.
- **Administrative Efficiency:**
  - Streamlines tasks like scheduling, billing, and record-keeping.

### **Challenges & Concerns:**
- **Data Privacy:** Risks of unauthorized access or misuse of patient data.
- **Bias in Datasets:** AI models may produce unfair outcomes if trained on incomplete or biased data.
- **Ethical & Safety Issues:**
  - Need for human oversight to ensure patient-centered care.
  - Requires validation to prevent errors in diagnosis/treatment.

### **Key Takeaway:**
AI has **huge potential** to revolutionize

# Conditional Chains

In [10]:
model= ChatMistralAI(model='mistral-medium-2508')

parser1= StrOutputParser()

class feedback(BaseModel):
    
    sentiment: Literal['positive', 'negative']= Field(description='give the sentiment of the feedback text')
    
parser2= PydanticOutputParser(pydantic_object=feedback)

prompt1= PromptTemplate(
    template= 'Classify the sentiment of the following feedback text into positive or negative\n{feedback}\n{format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction': parser2.get_format_instructions}
)
prompt2= PromptTemplate(
    template='Write only an appropriate response to this positive feedback\n{feedback}',
    input_variables=['feedback']
)
prompt3= PromptTemplate(
    template='Write only an appropriate response to this negative feedback\n{feedback}',
    input_variables=['feedback']
)

classifier_chain= prompt1 | model | parser2

branch_chain= RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt2 | model | parser1),
    (lambda x: x.sentiment == 'negative', prompt3 | model | parser1),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain= classifier_chain | branch_chain

result= chain.invoke({'feedback': 'Wow! this phone lasts for 1 day on full charge.'})
print(result)

"Thank you so much for your kind words! I really appreciate your positive feedback—it means a lot!"
